# Step 0.1.b — Check of the SVM background: MIRCA2000 cropland vs ESA CCI Land Cover 2000

The candidate background of the SVM (absence class) is the **MIRCA2000 cropland**: land cells with any harvested area
of any of the 26 MIRCA2000 classes (notebook 0.1, `mirca2000_any_crop_ecearth3.nc`), a target-group background
(absences only where other crops are grown, so that an absence reflects the crop and not the land use). MIRCA2000 is
built on the cropland extent of Ramankutty et al. (2008), the 'arable land' used as background by Ceglar et al. (2021).

Independent check with the **ESA CCI Land Cover map of 2000** (v2.0.7, 300 m, from satellite; BDAP
`base/Landcover/GLOBAL/ESA-CCI-LC`): share of each EC-Earth3 cell classified as cropland,
- classes 10, 11, 12 (rainfed cropland), 20 (irrigated), 30 (mosaic, cropland > 50 %): `crop`;
- the same + class 40 (mosaic, natural vegetation > 50 %, cropland < 50 %): `crop+mosaic`;

relative to the cell's land pixels (no data 0 and water 210 excluded). As for MIRCA2000, no threshold: a cell is ESA
cropland if its share is > 0. Outputs in `<out_root>/0_masks_ecearth3/`: `esa_cci_lc_2000_cropland_ecearth3.nc` (shares per cell),
`check_background_esa_cci_2000.csv` and the maps.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time
from pathlib import Path
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb
OUT = cu.out_dir(CFG, cu.MASKS)

CROP_CLASSES = [10, 11, 12, 20, 30]      # ESA CCI LCCS cropland classes
MOSAIC_CLASSES = [40]                    # mosaic natural vegetation (> 50 %) / cropland (< 50 %)
NOT_LAND = [0, 210]                      # no data, water
THRESHOLDS = [0.0]                       # an EC-Earth3 cell is ESA cropland if its share > threshold (no threshold,
                                         # as for MIRCA2000; e.g. [0.0, 0.01, 0.05] to compare)
BLOCK = 1800                             # rows of the 300 m map read at once (1800 x 129600 bytes ~ 230 MB)

GRID = cu.svm_grid(CFG)
NLAT, NLON = GRID["lat"].size, GRID["lon"].size
LAND = cu.land_mask(CFG).values
mirca = xr.open_dataset(OUT / "mirca2000_any_crop_ecearth3.nc")
MIRCA = (mirca["cropland"].values == 1) & LAND
LC_FILE = cu.data_file(CFG, "esa_cci_lc_2000", "lccs_class")
print("ESA CCI file:", LC_FILE)
print(f"land cells {int(LAND.sum())}, MIRCA2000 cropland cells {int(MIRCA.sum())}")

## 1. ESA CCI 2000: cropland share of each EC-Earth3 cell (300 m pixels counted per cell)

In [ ]:
t0 = time.time()
ds = xr.open_dataset(LC_FILE)
lc = ds["lccs_class"]
if "time" in lc.dims:
    lc = lc.isel(time=0)
lat, lon = ds["lat"].values, ds["lon"].values
I_row, _ = cu.cell_of(lat, np.zeros_like(lat), GRID)          # EC-Earth3 row of each 300 m row
_, J_col = cu.cell_of(np.zeros_like(lon), lon, GRID)          # EC-Earth3 column of each 300 m column
n_land = np.zeros(NLAT * NLON); n_crop = np.zeros(NLAT * NLON); n_mos = np.zeros(NLAT * NLON)
for r0 in range(0, lat.size, BLOCK):
    x = np.asarray(lc.isel(lat=slice(r0, r0 + BLOCK)).values)
    is_land = ~np.isin(x, NOT_LAND)
    is_crop = np.isin(x, CROP_CLASSES)
    is_mos = np.isin(x, MOSAIC_CLASSES)
    for r in range(x.shape[0]):
        k = I_row[r0 + r] * NLON + J_col
        n_land += np.bincount(k, weights=is_land[r], minlength=NLAT * NLON)
        n_crop += np.bincount(k, weights=is_crop[r], minlength=NLAT * NLON)
        n_mos += np.bincount(k, weights=is_mos[r], minlength=NLAT * NLON)
    if r0 // BLOCK % 6 == 0:
        print(f"  rows {r0 + x.shape[0]} / {lat.size} ({(time.time() - t0) / 60:.1f} min)", flush=True)
ds.close()
share_crop = (n_crop / np.maximum(n_land, 1)).reshape(NLAT, NLON)
share_cm = ((n_crop + n_mos) / np.maximum(n_land, 1)).reshape(NLAT, NLON)
xr.Dataset({"share_crop": (("lat", "lon"), share_crop.astype("float32"), {"long_name": "ESA CCI 2000 cropland share (classes 10-30) of the cell's land pixels"}),
            "share_crop_mosaic": (("lat", "lon"), share_cm.astype("float32"), {"long_name": "same with class 40 (mosaic, cropland < 50 %)"})},
           coords={"lat": GRID["lat"], "lon": GRID["lon"]}).to_netcdf(OUT / "esa_cci_lc_2000_cropland_ecearth3.nc")
print(f"done in {(time.time() - t0) / 60:.1f} min")

## 2. Agreement with the MIRCA2000 cropland (land cells)

For each definition and threshold: cells cropland in both, in MIRCA2000 only, in ESA CCI only, agreement (% of land
cells), and the share of MIRCA2000's harvested area that falls in ESA cropland cells.

In [ ]:
AREA = np.nan_to_num(mirca["area_ha"].values)
rows = []
for name, share in (("crop", share_crop), ("crop+mosaic", share_cm)):
    for thr in THRESHOLDS:
        esa = (share > thr) & LAND
        rows.append({"ESA definition": name, "share >": thr,
                     "both": int((esa & MIRCA).sum()), "MIRCA only": int((MIRCA & ~esa).sum()),
                     "ESA only": int((esa & ~MIRCA).sum()), "neither": int((LAND & ~esa & ~MIRCA).sum()),
                     "agreement (% of land)": round(100 * ((esa == MIRCA) & LAND).sum() / LAND.sum(), 1),
                     "MIRCA area in ESA cells (%)": round(100 * AREA[esa].sum() / AREA[LAND].sum(), 1)})
agree = pd.DataFrame(rows).set_index(["ESA definition", "share >"])
agree.to_csv(OUT / "check_background_esa_cci_2000.csv")
display(agree)

# presence cells of each crop that fall outside the ESA cropland (classes 10-30, share > 0)
esa0 = (share_crop > 0) & LAND
rows = []
for crop in ("Maize_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf", "Potato_rf", "Rice_rf", "Soybean_rf"):
    p = cu.crop_mask(CFG, crop)["presence"].values == 1
    rows.append({"crop": crop, "presence cells": int(p.sum()), "outside MIRCA cropland": int((p & ~MIRCA).sum()),
                 "outside ESA cropland": int((p & ~esa0).sum())})
display(pd.DataFrame(rows).set_index("crop"))

## 3. Maps: ESA cropland share, and where the two backgrounds disagree

In [ ]:
coords = {"lat": GRID["lat"], "lon": GRID["lon"]}
fig, axes = plt.subplots(2, 1, figsize=(16, 13))
xr.DataArray(np.where(LAND, share_crop, np.nan), coords=coords, dims=("lat", "lon")).plot(
    ax=axes[0], cmap="YlGn", vmin=0, vmax=0.5, cbar_kwargs={"label": "share of the cell"})
axes[0].set_title("ESA CCI 2000: cropland share (classes 10-30) of each EC-Earth3 land cell")
esa = (share_crop > 0) & LAND
cat = np.where(~LAND, np.nan, np.where(esa & MIRCA, 3, np.where(MIRCA, 2, np.where(esa, 1, 0))))
from matplotlib.colors import ListedColormap
xr.DataArray(cat, coords=coords, dims=("lat", "lon")).plot(
    ax=axes[1], cmap=ListedColormap(["#eeeeee", "#2b83ba", "#d7191c", "#1a9641"]), vmin=-0.5, vmax=3.5,
    cbar_kwargs={"ticks": [0, 1, 2, 3], "label": "0 neither, 1 ESA only, 2 MIRCA only, 3 both"})
axes[1].set_title("Background: MIRCA2000 cropland vs ESA CCI 2000 cropland (share > 0)")
plt.tight_layout(); plt.savefig(OUT / "check_background_esa_cci_2000.png", dpi=90); plt.show()